# Milestone 1: dataset preparation, text embeddings and first samples (Google Colab)

Runtime → Change runtime type → **T4 GPU** (or better). Data and outputs are kept on Google Drive in `MyDrive/IE7615_Project2`, so a disconnected session can resume.

| Step | Script | Output |
|---|---|---|
| 1 | `01_prepare_data.py` | COCO val2017 subset, 512×512 images, `results/dataset_stats.json` |
| 2 | `02_encode_captions.py` | CLIP ViT-L/14 embeddings for every caption |
| 3 | `03_embedding_checks.py` | caption similarity + text→image retrieval, `results/embedding_checks.json` |
| 4 | `04_sample_prompts.py` | 5 prompt→image samples, external-embedding conditioning check |

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/IE7615_Project2'
except Exception as e:
    print('Google Drive not mounted, outputs stay in /content/outputs:', e)
    DRIVE = '/content/outputs'
REPO = 'https://github.com/anshuman-dataverse/generative-deep-learning-project.git'
NUM_IMAGES = 5000
os.makedirs(f'{DRIVE}/data', exist_ok=True)

In [ ]:
%cd /content
!test -d project || git clone -q $REPO project
!cd project && git pull -q
!pip -q install -r project/requirements.txt
# keep the COCO download and processed data on Drive
!rm -rf project/milestone1/data && ln -sfn "$DRIVE/data" project/milestone1/data

## 1. Download COCO val2017, clean captions, subset and resize

In [ ]:
%cd /content/project/milestone1/scripts
!mkdir -p ../runs
!python 01_prepare_data.py --num-images $NUM_IMAGES 2>&1 | tee ../runs/01_prepare_data.log

In [ ]:
from IPython.display import Image, display
display(Image('../screenshots/dataset_examples.png'), Image('../screenshots/caption_lengths.png'))

## 2. Text → embedding pipeline (CLIP ViT-L/14)

In [ ]:
!python 02_encode_captions.py 2>&1 | tee ../runs/02_encode_captions.log

In [ ]:
import sys, torch
sys.path.insert(0, '.')
from t2i.embeddings import TextEncoder
enc = TextEncoder('openai/clip-vit-large-patch14')
caption = 'A man riding a bicycle down a city street.'
tok = enc.tokenize([caption])
print(enc.tokenizer.convert_ids_to_tokens(tok['input_ids'][0][:int(tok['attention_mask'].sum())]))
out = enc.encode([caption])
print('per-token (U-Net cross-attention context):', tuple(out['hidden'].shape))
print('pooled (EOS token):', tuple(out['pooled'].shape))
del enc; torch.cuda.empty_cache()

## 3. Embedding sanity checks

In [ ]:
!python 03_embedding_checks.py 2>&1 | tee ../runs/03_embedding_checks.log

In [ ]:
display(Image('../screenshots/caption_similarity.png'), Image('../screenshots/retrieval_examples.png'))

## 4. Five prompt → image test runs (Stable Diffusion v1.5, DDIM, 30 steps, guidance 7.5)

In [ ]:
!python 04_sample_prompts.py 2>&1 | tee ../runs/04_sample_prompts.log

In [ ]:
display(Image('../screenshots/milestone1_samples.png'), Image('../screenshots/conditioning_check.png'))

## 5. Save outputs to Drive
Download `milestone1_outputs.zip`, unzip into `milestone1/` of the local repo and commit.

In [ ]:
%cd /content/project/milestone1
!zip -qr "$DRIVE/milestone1_outputs.zip" results screenshots runs
!ls -la "$DRIVE"